# 00 — Colab Setup

Run this notebook first to validate the environment. **Each pipeline notebook (01, 02, …) includes its own identical setup cell** because Colab tabs and runtimes do not share state.


## Why separate setup in every notebook?

- Opening `01` or `02` in a **new Colab tab** starts a **fresh runtime** — `00` setup is not inherited.
- `/content` is **temporary**; disconnects can wipe local outputs.
- **Google Drive** (`USE_GOOGLE_DRIVE=True`) stores `data/`, `reports/`, and `artifacts/` at:
  `/content/drive/MyDrive/openf1_big_data_pipeline`
- **Code** stays in `/content/openf1-big-data-pipeline` (GitHub clone).
- Set `OPENF1_DATA_ROOT` **before** importing `openf1_pipeline.config`.


## Standard Colab setup


In [2]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Verify OpenF1 client


In [3]:
import pandas as pd
import requests

from openf1_pipeline.config import ENDPOINTS, OPENF1_BASE_URL, SEASONS
from openf1_pipeline.ingestion.openf1_client import OpenF1Client

print("SEASONS:", SEASONS)
print("ENDPOINTS:", ENDPOINTS)
print("OpenF1 base URL:", OPENF1_BASE_URL)
print("OpenF1Client:", OpenF1Client().base_url)
print("Setup complete — proceed to 01_ingestion_bronze.ipynb")


SEASONS: [2023, 2024, 2025]
ENDPOINTS: ['meetings', 'sessions', 'drivers', 'laps', 'pit', 'weather', 'position', 'race_control', 'session_result', 'starting_grid']
OpenF1 base URL: https://api.openf1.org/v1
OpenF1Client: https://api.openf1.org/v1
Setup complete — proceed to 01_ingestion_bronze.ipynb
